# 01 EDA Template

목표: 데이터 구조, target 분포, 결측치, 이상치, 데이터 누수 후보를 확인하고 `reports/DATA_CARD.md`에 옮길 근거를 만듭니다.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/raw/dataset.csv')
TARGET = 'label'

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    from sklearn.datasets import load_breast_cancer
    demo = load_breast_cancer(as_frame=True)
    df = demo.frame.rename(columns={'target': TARGET})

df.head()

## 1. 기본 구조

In [ ]:
display(df.shape)
display(df.info())
display(df.describe(include='all').T.head(30))

## 2. 결측치와 중복

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
display(missing[missing > 0])
print('duplicated rows:', df.duplicated().sum())

## 3. Target 분포

In [ ]:
display(df[TARGET].value_counts(dropna=False))
sns.countplot(data=df, x=TARGET)
plt.title('Target distribution')
plt.show()

## 4. Feature 분포와 누수 후보

- target을 직접 계산한 컬럼이 있는가?
- 시간 순서상 미래 정보를 담은 컬럼이 있는가?
- ID, 이름, 결과 후 생성되는 상태값이 feature에 섞였는가?

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.drop(TARGET, errors='ignore')
df[numeric_cols[:12]].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()

corr_with_target = df[numeric_cols].corrwith(df[TARGET]).abs().sort_values(ascending=False)
display(corr_with_target.head(20))

## 5. Split 점검 메모

- stratify가 필요한가?
- 같은 사용자/상품/문서가 train과 test에 동시에 들어가면 안 되는가?
- 시간 기반 split이 더 적절한가?

이 메모를 `reports/DATA_CARD.md`와 `data_manifest.json`에 반영하세요.